# DroneTask — Mouse-Dynamics QC Module: Pilot Analysis

**What this is.** An *exploratory mouse-dynamics quality-control measure*, acquired with the fixed 8-target click paradigm used by Shen et al. (2014) and Acien et al. (2022, *BeCAPTCHA-Mouse*), appended after the last memory probe. It is **not** a validated bot classifier, and no participant is excluded on the basis of anything computed here.

**Purpose of this notebook.** Decide whether the optional measure behaves sensibly in the DroneTask/Prolific population before investing anything further in it. The core task and splitter do not depend on this notebook. Every section maps to a go/no-go criterion (thresholds live in `CRITERIA` below and are meant to be agreed on *before* looking at the pilot data):

| Check | Question | Proposed go criterion |
|---|---|---|
| **Completion & devices** | Do sessions assigned/requested to run QC finish the 8 targets? | ≥ 90 % status = `complete`; an expected session with no summary counts as incomplete |
| **Sampling quality** | Are completed trajectories sampled densely enough to compute features? | ≥ 80 % of completed participants: median ≥ 20 points / trajectory and median Δt ≤ 20 ms |
| **Feature sanity** | Do the six kinematic features vary with the geometry as they should? | Path length & duration increase with nominal length; efficiency < 1 |
| **Within-person stability** | Are per-participant medians well-defined (IQR not dominated by outliers)? | Descriptive |
| **Synthetic separation** | Can the same feature extractor tell obvious synthetic paths (straight + constant speed) from pilot humans? | Separation index ≥ 0.8 on efficiency **or** turning angle for type A |
| **Sampling sensitivity** | Are feature values driven by sampling covariates (browser/device) rather than behaviour? | \|Spearman ρ\| < 0.5 between turning angle and median Δt |

**Analysis populations.** The completion audit includes every metadata session marked as QC-requested/assigned, including asset failures and missing summaries. Primary kinematic analyses include only rows with `trajectory_completed == true` from sessions whose summary status is `complete` and which contain exactly one completed row for each `trajectory_id` 1–8. When the summary reports a completed count, it must also equal 8. Skipped, incomplete, duplicated-ID, and missing sessions remain in a separate audit table.

**Inputs.** `data/clean/*_mouse_trajectory.csv` (raw samples as JSON), `*_mouse_summary.csv`, and `*_metadata.csv`, produced by `analysis/split_drone_data.py`.

## Setup & data loading

In [ ]:
import os, glob, json, re, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Work whether Jupyter was launched from the repository root or this folder.
_cwd = os.path.abspath(os.getcwd())
_candidates = [_cwd, os.path.join(_cwd, 'mouse_qc'), os.path.join(_cwd, 'analysis', 'mouse_qc')]
NOTEBOOK_DIR = next((p for p in _candidates if os.path.isfile(os.path.join(p, 'mouse_features.py'))), _cwd)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
from mouse_features import (GEOMETRY_G1, CORE_FEATURES, SUPP_FEATURES, FEATURE_LABELS, SYNTH_KINDS,
                            movement_table, trajectory_points, compute_features, features_from_table,
                            summarise_by_participant, build_synthetic_set, rank_auc)

REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind', font_scale=1.05)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)

CLEAN_DIR = os.path.abspath(os.environ.get('DRONE_CLEAN_DIR', os.path.join(REPO_ROOT, 'data', 'clean')))
RNG = np.random.default_rng(20260929)

# ── Pre-specified decision thresholds (edit BEFORE looking at pilot data) ──
CRITERIA = {
    'min_complete_rate':      0.90,   # complete / all metadata sessions assigned or requesting QC
    'min_points_median':      20,     # median points per trajectory, per completed participant
    'max_dt_median_ms':       20.0,   # median inter-sample interval, per completed participant
    'min_sampling_pass_rate': 0.80,   # fraction of completed participants meeting both checks
    'min_separation_typeA':   0.80,   # separation index 2|AUC-0.5| for human vs type-A synthetic
    'max_abs_rho_turn_dt':    0.50,   # |Spearman rho| turning angle vs median dt
    'n_synth_per_kind':       30,     # synthetic pseudo-participants per kind
}
MOVEMENTS = movement_table()
print('clean data:', CLEAN_DIR)
print(MOVEMENTS.to_string(index=False))

In [ ]:
def session_from_filename(path, table_name):
    """Stable split-session key shared by metadata, summary, and trajectory files."""
    base = os.path.basename(path)
    suffix = f'_{table_name}.csv'
    return base[:-len(suffix)] if base.endswith(suffix) else os.path.splitext(base)[0]

def participant_from_filename(path, table_name):
    """Recover IDs from legacy split files that lack participant_id."""
    return re.sub(r'_(?:\d{4}-\d{2}-\d{2}|nodatetime)$', '',
                  session_from_filename(path, table_name))

def load_tables(pattern, table_name):
    files = sorted(glob.glob(os.path.join(CLEAN_DIR, pattern)))
    frames = []
    for f in files:
        d = pd.read_csv(f)
        if 'participant_id' not in d.columns:
            d.insert(0, 'participant_id', participant_from_filename(f, table_name))
        d['session_key'] = session_from_filename(f, table_name)
        d['_source_file'] = os.path.basename(f)
        frames.append(d)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

traj = load_tables('*_mouse_trajectory.csv', 'mouse_trajectory')
summ = load_tables('*_mouse_summary.csv', 'mouse_summary')
meta = load_tables('*_metadata.csv', 'metadata')
print(f'{traj.participant_id.nunique() if len(traj) else 0} participants with trajectories, '
      f'{len(traj)} trajectory rows, {len(summ)} summary rows, {len(meta)} metadata rows')

# Geometry cross-check: the data are self-describing; analyse versions separately if this fails.
if len(summ) and 'mt_geometry_json' in summ.columns:
    versions = summ['mt_geometry_version'].dropna().unique()
    print('geometry versions in data:', versions)
    for gj in summ['mt_geometry_json'].dropna().unique():
        g = json.loads(gj)
        same = (g['start'] == GEOMETRY_G1['start'] and g['targets'] == GEOMETRY_G1['targets']
                and abs(g['target_radius_frac'] - GEOMETRY_G1['target_radius_frac']) < 1e-9)
        print('  matches GEOMETRY_G1:', same, '| version', g.get('version'))
        assert same, 'Geometry in data differs from mouse_features.GEOMETRY_G1 — analyse by version.'

if len(traj):
    traj['trajectory_id'] = pd.to_numeric(traj['trajectory_id'], errors='coerce').astype('Int64')
    traj = traj.merge(
        MOVEMENTS[['trajectory_id', 'nominal_length', 'movement_orientation', 'movement_length_class']],
        on='trajectory_id', how='left', suffixes=('', '_geom')
    )
traj.head(3)

## Completion, exclusion, and device audit

**Denominator.** Every metadata session for which `mouse_qc_requested` or `mouse_qc_enabled` is true, or whose `task_variant` identifies a mouse-QC variant. A legacy summary without those newer metadata fields is also included. Thus an assigned session whose asset failed or whose summary is missing remains in the denominator.

**Primary analysis population.** A session must have `mt_status == 'complete'` and exactly one completed row for every `trajectory_id` 1–8 (no duplicates, omissions, or unexpected IDs). When `mt_n_trajectories_completed` is available in the summary, it must also equal 8. All other sessions/rows are retained below as an explicit audit, not silently dropped.

**Go criterion.** ≥ 90 % of expected sessions have a present summary with `mt_status == 'complete'`.

In [ ]:
TRUE_VALUES = {'1', 'true', 't', 'yes', 'y', 'on'}

def scalar_bool(value):
    if value is None or value is pd.NA:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return value != 0
    return str(value).strip().lower() in TRUE_VALUES

def bool_column(frame, name):
    if name not in frame.columns:
        return pd.Series(False, index=frame.index, dtype=bool)
    return frame[name].map(scalar_bool).astype(bool)

def col_or_na(frame, name):
    return frame[name] if name in frame.columns else pd.Series(pd.NA, index=frame.index)

meta_audit = meta.copy()
if len(meta_audit):
    variant = col_or_na(meta_audit, 'task_variant').fillna('').astype(str)
    meta_audit['qc_expected'] = (
        bool_column(meta_audit, 'mouse_qc_requested') |
        bool_column(meta_audit, 'mouse_qc_enabled') |
        variant.str.contains(r'mouse[-_ ]?qc', case=False, regex=True)
    )
else:
    meta_audit = pd.DataFrame(columns=['session_key', 'participant_id', 'qc_expected'])

expected_meta = meta_audit[meta_audit['qc_expected']].copy()
summ_audit = summ.copy()
if len(summ_audit):
    # A split session should have one summary; retain the last and expose duplicates.
    duplicate_summaries = summ_audit.duplicated('session_key', keep=False)
    if duplicate_summaries.any():
        print('WARNING: duplicate summary sessions:', sorted(summ_audit.loc[duplicate_summaries, 'session_key'].astype(str).unique()))
    summ_audit = summ_audit.drop_duplicates('session_key', keep='last')
else:
    summ_audit = pd.DataFrame(columns=['session_key', 'participant_id'])

audit = expected_meta.merge(
    summ_audit, on='session_key', how='outer',
    suffixes=('_meta', '_summary'), indicator='summary_merge'
)
participant_from_summary = col_or_na(audit, 'participant_id_summary').combine_first(
    col_or_na(audit, 'participant_id')
)
audit['participant_id'] = participant_from_summary.combine_first(
    col_or_na(audit, 'participant_id_meta')
)
audit['qc_expected'] = audit['qc_expected'].fillna(True).astype(bool)
audit['summary_present'] = audit['summary_merge'].isin(['both', 'right_only'])
status_from_summary = col_or_na(audit, 'mt_status_summary').combine_first(
    col_or_na(audit, 'mt_status')
)
audit['mt_status'] = status_from_summary.combine_first(col_or_na(audit, 'mt_status_meta'))
device_from_summary = col_or_na(audit, 'mt_pointer_type_self_report_summary').combine_first(
    col_or_na(audit, 'mt_pointer_type_self_report')
)
audit['mt_pointer_type_self_report'] = device_from_summary.combine_first(
    col_or_na(audit, 'mt_pointer_type_self_report_meta')
)
audit['session_completed'] = audit['summary_present'] & audit['mt_status'].eq('complete')
audit['audit_status'] = audit['mt_status'].fillna('missing_summary')
missing_summary = ~audit['summary_present']
if 'mouse_qc_asset_status' in audit.columns:
    reason = audit['mouse_qc_asset_status'].fillna('unknown').astype(str)
    audit.loc[missing_summary, 'audit_status'] = 'missing_summary:' + reason[missing_summary]

# Verify row-level completeness independently of the summary status. A
# raw count of 8 is insufficient: duplicated IDs can conceal a missing ID.
EXPECTED_TRAJECTORY_IDS = set(range(1, 9))
trajectory_audit_columns = [
    'session_key', 'n_completed_rows', 'n_unique_completed_ids',
    'n_invalid_completed_ids',
    'completed_id_set', 'missing_completed_ids',
    'duplicate_completed_ids', 'unexpected_completed_ids',
    'completed_ids_exact',
]
if len(traj):
    traj['_trajectory_completed_bool'] = bool_column(traj, 'trajectory_completed')
    trajectory_audit_rows = []
    completed = traj[traj['_trajectory_completed_bool']]
    for session_key, group in completed.groupby('session_key', dropna=False):
        parsed_ids = pd.to_numeric(group['trajectory_id'], errors='coerce')
        n_invalid = int(parsed_ids.isna().sum())
        completed_ids = parsed_ids.dropna().astype(int).tolist()
        id_counts = pd.Series(completed_ids, dtype='int64').value_counts()
        observed = set(completed_ids)
        missing = sorted(EXPECTED_TRAJECTORY_IDS - observed)
        duplicates = sorted(int(i) for i, n in id_counts.items() if n > 1)
        unexpected = sorted(observed - EXPECTED_TRAJECTORY_IDS)
        exact = (len(group) == 8 and n_invalid == 0
                 and observed == EXPECTED_TRAJECTORY_IDS
                 and not duplicates and not unexpected)
        trajectory_audit_rows.append({
            'session_key': session_key,
            'n_completed_rows': len(group),
            'n_unique_completed_ids': len(observed),
            'n_invalid_completed_ids': n_invalid,
            'completed_id_set': ','.join(map(str, sorted(observed))),
            'missing_completed_ids': ','.join(map(str, missing)),
            'duplicate_completed_ids': ','.join(map(str, duplicates)),
            'unexpected_completed_ids': ','.join(map(str, unexpected)),
            'completed_ids_exact': exact,
        })
    trajectory_audit = pd.DataFrame(trajectory_audit_rows, columns=trajectory_audit_columns)
else:
    traj['session_key'] = pd.Series(dtype=object)
    traj['participant_id'] = pd.Series(dtype=object)
    traj['_trajectory_completed_bool'] = pd.Series(dtype=bool)
    trajectory_audit = pd.DataFrame(columns=trajectory_audit_columns)
audit = audit.merge(trajectory_audit, on='session_key', how='left')
audit['n_completed_rows'] = audit['n_completed_rows'].fillna(0).astype(int)
audit['n_unique_completed_ids'] = audit['n_unique_completed_ids'].fillna(0).astype(int)
audit['n_invalid_completed_ids'] = audit['n_invalid_completed_ids'].fillna(0).astype(int)
audit['completed_id_set'] = audit['completed_id_set'].fillna('')
audit['missing_completed_ids'] = audit['missing_completed_ids'].fillna('1,2,3,4,5,6,7,8')
audit['duplicate_completed_ids'] = audit['duplicate_completed_ids'].fillna('')
audit['unexpected_completed_ids'] = audit['unexpected_completed_ids'].fillna('')
audit['completed_ids_exact'] = audit['completed_ids_exact'].fillna(False).astype(bool)
summary_completed_count = pd.to_numeric(col_or_na(audit, 'mt_n_trajectories_completed'), errors='coerce')
audit['summary_completed_count_ok'] = summary_completed_count.isna() | summary_completed_count.eq(8)
audit['primary_session'] = (audit['session_completed'] & audit['completed_ids_exact'] &
                            audit['summary_completed_count_ok'])
audit['primary_exclusion_reason'] = np.select(
    [~audit['session_completed'], ~audit['completed_ids_exact'],
     ~audit['summary_completed_count_ok']],
    ['session_incomplete_or_missing_summary', 'completed_trajectory_id_set_invalid',
     'summary_completed_count_not_8'],
    default='',
)

complete_rate = audit['session_completed'].mean() if len(audit) else np.nan
print(f"complete rate = {complete_rate:.0%}  (criterion ≥ {CRITERIA['min_complete_rate']:.0%}; denominator N={len(audit)})")
print('audit status counts:', audit['audit_status'].value_counts(dropna=False).to_dict())
print('self-reported device:', audit['mt_pointer_type_self_report'].value_counts(dropna=False).to_dict())
GO_COMPLETE = bool(np.isfinite(complete_rate) and complete_rate >= CRITERIA['min_complete_rate'])

audit_cols = [
    'session_key', 'participant_id', 'task_variant', 'mouse_qc_requested', 'mouse_qc_enabled',
    'mouse_qc_asset_status', 'summary_present', 'audit_status',
    'n_completed_rows', 'n_unique_completed_ids', 'n_invalid_completed_ids',
    'completed_id_set', 'missing_completed_ids', 'duplicate_completed_ids',
    'unexpected_completed_ids', 'summary_completed_count_ok',
    'primary_exclusion_reason', 'primary_session', 'mt_pointer_type_self_report',
    'mt_n_trajectories_completed', 'mt_coalesced_supported',
    'mt_area_side_px', 'mt_device_pixel_ratio', 'mt_total_misclicks',
]
display(audit[[c for c in audit_cols if c in audit.columns]])

primary_sessions = set(audit.loc[audit['primary_session'], 'session_key'])
primary_mask = traj['_trajectory_completed_bool'] & traj['session_key'].isin(primary_sessions)
primary_traj = traj[primary_mask].copy()
primary_traj['source_participant_id'] = primary_traj['participant_id']
primary_traj['participant_id'] = primary_traj['session_key']  # analysis unit = session
excluded_traj = traj[~primary_mask].copy()
if len(excluded_traj):
    excluded_traj['exclusion_reason'] = np.select(
        [
            ~excluded_traj['_trajectory_completed_bool'],
            ~excluded_traj['session_key'].isin(set(audit.loc[audit['session_completed'], 'session_key'])),
        ],
        ['trajectory_incomplete', 'session_incomplete_or_missing_summary'],
        default='completed_trajectory_id_set_or_summary_count_invalid',
    )
    display(excluded_traj.groupby(['session_key', 'participant_id', 'exclusion_reason'], dropna=False).size().rename('rows').reset_index())
print(f'primary trajectories: {len(primary_traj)} rows from {len(primary_sessions)} session(s); excluded/audit-only rows: {len(excluded_traj)}')
if not len(primary_traj):
    raise RuntimeError('No complete QC session has exactly one completed trajectory_id 1–8; review the audit table above.')

## Sampling quality

**Question.** Is the raw signal dense and regular enough to support the kinematic features?

**Population.** Only completed trajectory rows from complete 8-trajectory sessions (`primary_traj`). Expected-but-missing, skipped, partial-session, and incomplete-trajectory rows remain in the preceding audit and do not enter feature summaries.

**Method.** For every retained trajectory count the points used by the feature extractor (opening click + moves + closing click), the median inter-sample interval Δt, the fraction of Δt = 0 samples, and the share of samples from `getCoalescedEvents()`. Summarise per participant.

**Go criterion.** ≥ 80 % of completed participants have median points ≥ 20 and median Δt ≤ 20 ms. High `frac_zero_dt` is a flag to inspect, not an exclusion.

In [ ]:
feat = features_from_table(
    primary_traj, id_cols=('participant_id', 'source_participant_id', 'trajectory_id')
)
feat['coalesced_share'] = (
    pd.to_numeric(primary_traj['n_coalesced_samples'], errors='coerce') /
    pd.to_numeric(primary_traj['n_samples'], errors='coerce').replace(0, np.nan)
).values
samp = (feat.groupby('participant_id')
            .agg(n_traj=('trajectory_id', 'size'),
                 points_median=('n_points', 'median'), points_min=('n_points', 'min'),
                 dt_median=('median_dt_ms', 'median'), zero_dt_frac=('frac_zero_dt', 'median'),
                 coalesced_share=('coalesced_share', 'median'))
            .reset_index())
samp['sampling_pass'] = (samp['points_median'] >= CRITERIA['min_points_median']) & (samp['dt_median'] <= CRITERIA['max_dt_median_ms'])
display(samp.round(3))
pass_rate = samp['sampling_pass'].mean() if len(samp) else np.nan
print(f"sampling pass rate = {pass_rate:.0%}  (criterion ≥ {CRITERIA['min_sampling_pass_rate']:.0%})")
GO_SAMPLING = bool(np.isfinite(pass_rate) and pass_rate >= CRITERIA['min_sampling_pass_rate'])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(feat['n_points'], bins=25, ax=axes[0]); axes[0].axvline(CRITERIA['min_points_median'], color='k', ls='--')
axes[0].set_title('Points per completed trajectory'); axes[0].set_xlabel('n points')
sns.histplot(feat['median_dt_ms'].dropna(), bins=25, ax=axes[1]); axes[1].axvline(CRITERIA['max_dt_median_ms'], color='k', ls='--')
axes[1].set_title('Median Δt per completed trajectory'); axes[1].set_xlabel('ms')
sns.histplot(feat['frac_zero_dt'].dropna(), bins=20, ax=axes[2]); axes[2].set_title('Fraction Δt = 0'); axes[2].set_xlabel('fraction')
plt.tight_layout(); plt.show()

## Figure 1 — completed raw trajectories on the target geometry

The primary plot contains only completed paths from complete 8-trajectory sessions. Skipped/incomplete paths are deliberately kept out of the feature plots and retained in `excluded_traj` for separate inspection. Look for (i) paths that reach every target, (ii) expected curvature/overshoot of human aiming, (iii) suspiciously straight or identical paths, and (iv) device-related artefacts.

In [ ]:
def draw_geometry(ax, label_targets=True):
    g = GEOMETRY_G1
    ax.add_patch(plt.Circle((g['start']['x'], g['start']['y']), g['start_radius_frac'],
                            fill=False, ec='#059669', lw=2, ls='--'))
    for i, tg in enumerate(g['targets']):
        ax.add_patch(plt.Circle((tg['x'], tg['y']), g['target_radius_frac'], fc='#dbeafe', ec='#1d4ed8', lw=1.5))
        if label_targets:
            ax.text(tg['x'], tg['y'], str(i + 1), ha='center', va='center', fontsize=9, color='#1e3a8a', fontweight='bold')
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(1.02, -0.02); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])

pal = sns.color_palette('husl', 8)
def plot_paths(ax, table, color_by='trajectory_id', alpha=0.45, lw=1.2, color=None):
    for _, r in table.iterrows():
        t, x, y = trajectory_points(r['samples'], r['events'])
        if len(t) < 2: continue
        c = color if color is not None else pal[int(r['trajectory_id']) - 1]
        ax.plot(x, y, color=c, alpha=alpha, lw=lw)

pids = sorted(primary_traj['participant_id'].unique())
n = len(pids); ncols = min(4, n); nrows = int(np.ceil(n / ncols))
fig = plt.figure(figsize=(6 + 3.2 * ncols, max(6, 3.2 * nrows)))
gs = fig.add_gridspec(nrows, ncols + 2)
ax0 = fig.add_subplot(gs[:, :2]); draw_geometry(ax0); plot_paths(ax0, primary_traj)
ax0.set_title(f'Completed sessions (N = {n}), coloured by movement')
for k, pid in enumerate(pids):
    ax = fig.add_subplot(gs[k // ncols, 2 + k % ncols]); draw_geometry(ax, label_targets=False)
    plot_paths(ax, primary_traj[primary_traj['participant_id'] == pid], alpha=0.9)
    dev = audit.loc[audit['session_key'] == pid, 'mt_pointer_type_self_report']
    ax.set_title(f"{pid}\n{dev.iloc[0] if len(dev) else ''}", fontsize=9)
plt.tight_layout(); plt.show()

## Core kinematic features (primary completed-session sample)

Skipped/incomplete rows and sessions without a complete summary are not included below.

**Definitions** (see `mouse_features.py`; no smoothing, no resampling):

| Feature | Formula | Note |
|---|---|---|
| duration $T$ | $t_{\text{hit}} - t_{\text{start}}$ | includes reaction/dwell before movement onset |
| path length $L$ | $\sum_i \|p_{i+1}-p_i\|$ | side units (task area side = 1) |
| displacement $D$ | $\|p_{\text{end}}-p_{\text{start}}\|$ | click-to-click, not centre-to-centre |
| mean speed | $L/T$ | side units · s⁻¹ |
| efficiency | $D/L$ | 1 = perfectly straight |
| mean \|turning angle\| | mean \|Δheading\| over non-zero segments | most sampling-sensitive feature |

**Question.** Do the features respond to the geometry as they should (longer nominal movements → longer $L$, longer $T$; efficiency < 1 for humans)?

In [ ]:
feat = feat.merge(primary_traj[['participant_id', 'trajectory_id', 'trajectory_duration_ms']], on=['participant_id', 'trajectory_id'], how='left')
# Cross-check: duration from samples == duration recorded by the plugin
if 'trajectory_duration_ms' in feat:
    d = (feat['duration_ms'] - feat['trajectory_duration_ms']).abs()
    print('max |duration(samples) − duration(plugin)| =', round(d.max(), 2), 'ms')
display(feat[['participant_id', 'trajectory_id', 'movement_label'] + CORE_FEATURES].round(4).head(10))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), CORE_FEATURES):
    sns.stripplot(data=feat, x='trajectory_id', y=f, ax=ax, alpha=.6, jitter=.2, color='#334155', size=4)
    sns.pointplot(data=feat, x='trajectory_id', y=f, ax=ax, estimator=np.median, errorbar=None, color='#dc2626', markers='_', scale=1.3, join=False)
    ax.set_title(FEATURE_LABELS[f]); ax.set_xlabel('movement (1–8)'); ax.set_ylabel('')
    labels = [f"{m}\n{l:.2f}" for m, l in zip(MOVEMENTS['trajectory_id'], MOVEMENTS['nominal_length'])]
    ax.set_xticklabels(labels, fontsize=8)
plt.suptitle('Per-trajectory features by movement (x tick = movement id / nominal length); red = median', y=1.01)
plt.tight_layout(); plt.show()

# Monotonic dependence of L and T on nominal length (Spearman across trajectories)
for f in ['path_length', 'duration_ms', 'displacement']:
    rho, p = stats.spearmanr(feat['nominal_length'], feat[f], nan_policy='omit')
    print(f'Spearman ρ(nominal_length, {f}) = {rho:.2f}  (p = {p:.3g})')
eff_med = feat['efficiency'].median()
print(f'median efficiency = {eff_med:.3f}; fraction of trajectories with efficiency ≥ 0.99: {(feat.efficiency >= 0.99).mean():.1%}')
GO_FEATURES = (stats.spearmanr(feat['nominal_length'], feat['path_length'], nan_policy='omit')[0] > 0.5) and (eff_med < 0.99)

## Within-person summaries (median and IQR across the 8 trajectories)

Per-participant medians are the unit of any later analysis; the IQR is the within-person variability. Bots and scripted input typically show near-zero within-person variability in efficiency and speed, so the IQR distribution is itself informative.

In [ ]:
pp = summarise_by_participant(feat)
display(pp.round(4))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, f in zip(axes.ravel(), CORE_FEATURES):
    sns.boxplot(data=feat, x='participant_id', y=f, ax=ax, color='#cbd5e1', fliersize=2, width=.6)
    ax.set_title(FEATURE_LABELS[f]); ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=60, labelsize=8)
plt.suptitle('Within-participant spread of each feature (8 trajectories per box)', y=1.01)
plt.tight_layout(); plt.show()

## Synthetic sanity checks (BeCAPTCHA-style baselines)

**Question.** If we run *exactly the same* feature extractor on trivially synthetic paths for the *same* eight target positions, do the human distributions separate from them?

**Method.** Following the shape × velocity-profile logic of Acien et al.'s function-based generator, build synthetic pseudo-participants (8 trajectories each) of three kinds, plus one stress test:

- **A** perfectly straight, constant speed
- **B** straight, acceleration/deceleration (minimum-jerk profile)
- **C** smoothly curved (quadratic Bézier), acceleration/deceleration
- **D** (stress test) C + Gaussian position jitter + Δt jitter — deliberately closer to human paths

Durations are drawn from the *pooled human duration distribution for the same movement*, so separation cannot be driven trivially by duration. Start/end points are target centres with a small click scatter. Sampling interval 8 ms (≈ 125 Hz, a typical USB mouse).

**Statistic.** For every feature and kind, the Mann–Whitney AUC of human vs synthetic values, reported as a **separation index** $2\,|\mathrm{AUC}-0.5| \in [0,1]$ (0 = indistinguishable by a single threshold, 1 = perfectly separable). This is a *descriptive* statistic, not a fitted classifier.

**Go criterion.** Separation index ≥ 0.8 on efficiency **or** turning angle for type A. If humans cannot be told apart from constant-speed straight lines, the module is not worth pursuing.

In [ ]:
kinds = ['A_linear_const', 'B_linear_accdec', 'C_curved_accdec', 'D_curved_accdec_noisy']
synth = build_synthetic_set(feat, n_per_kind=CRITERIA['n_synth_per_kind'], rng=RNG, kinds=kinds)
synth_feat = features_from_table(synth)
synth_feat = synth_feat.merge(synth[['participant_id', 'trajectory_id', 'source']].drop_duplicates(), on=['participant_id', 'trajectory_id'])
human_feat = feat.copy(); human_feat['source'] = 'human'
both = pd.concat([human_feat, synth_feat], ignore_index=True)
print(both.groupby('source').size())

# Overlay: retained human paths vs one synthetic pseudo-participant of each kind
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
kcol = dict(zip(kinds, sns.color_palette('deep', 4)))
for ax, kind in zip(axes, kinds):
    draw_geometry(ax); plot_paths(ax, primary_traj, alpha=.25, lw=1, color='#64748b')
    one = synth[synth['participant_id'] == f'{kind}_001']
    plot_paths(ax, one, alpha=.95, lw=2, color=kcol[kind]); ax.set_title(f'{kind}\n{SYNTH_KINDS[kind]}', fontsize=9)
plt.suptitle('Completed human paths (grey) vs one synthetic pseudo-participant per kind', y=1.04); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
order = ['human'] + kinds
for ax, f in zip(axes.ravel(), CORE_FEATURES):
    sns.violinplot(data=both, x='source', y=f, order=order, ax=ax, cut=0, inner='quartile', density_norm='width')
    ax.set_title(FEATURE_LABELS[f]); ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels(['human', 'A', 'B', 'C', 'D'])
plt.suptitle('Trajectory-level feature distributions: pilot humans vs synthetic baselines', y=1.01)
plt.tight_layout(); plt.show()

# Separation table (trajectory level)
rows = []
for kind in kinds:
    for f in CORE_FEATURES:
        h = human_feat[f].values; s_ = synth_feat.loc[synth_feat['source'] == kind, f].values
        auc = rank_auc(h, s_)
        rows.append({'kind': kind, 'feature': f,
                     'human_median': np.nanmedian(h), 'human_IQR': np.nanpercentile(h, 75) - np.nanpercentile(h, 25),
                     'synth_median': np.nanmedian(s_), 'synth_IQR': np.nanpercentile(s_, 75) - np.nanpercentile(s_, 25),
                     'AUC(human>synth)': auc, 'separation': 2 * abs(auc - 0.5)})
sep = pd.DataFrame(rows)
display(sep.pivot(index='feature', columns='kind', values='separation').loc[CORE_FEATURES].round(2))
display(sep[sep['kind'] == 'A_linear_const'].set_index('feature').round(4))

sepA = sep[(sep['kind'] == 'A_linear_const') & (sep['feature'].isin(['efficiency', 'mean_abs_turn_deg']))]['separation'].max()
print(f"best separation (type A, efficiency/turning) = {sepA:.2f}  (criterion ≥ {CRITERIA['min_separation_typeA']:.2f})")
GO_SEPARATION = sepA >= CRITERIA['min_separation_typeA']

### Participant-level view: within-person variability vs synthetic

The same comparison on the per-participant *IQR* (variability across the 8 movements). Scripted input has essentially no within-person variability in efficiency; humans do.

In [ ]:
pp_h = summarise_by_participant(human_feat); pp_h['source'] = 'human'
pp_s = []
for kind in kinds:
    p = summarise_by_participant(synth_feat[synth_feat['source'] == kind]); p['source'] = kind; pp_s.append(p)
pp_all = pd.concat([pp_h] + pp_s, ignore_index=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, f in zip(axes, ['efficiency', 'mean_abs_turn_deg', 'mean_speed']):
    sns.stripplot(data=pp_all, x='source', y=f + '_iqr', order=order, ax=ax, alpha=.7, jitter=.25)
    ax.set_title(f'Within-person IQR — {FEATURE_LABELS[f]}'); ax.set_xlabel(''); ax.set_ylabel('IQR'); ax.set_xticklabels(['human', 'A', 'B', 'C', 'D'])
plt.tight_layout(); plt.show()

## Sampling sensitivity

**Question.** Are feature values (especially turning angle) explained by sampling covariates that differ across browsers and devices rather than by behaviour? This is the main reason acceleration/jerk/Sigma-Lognormal features are deferred.

**Method.** Spearman correlations between each core feature and the sampling covariates (median Δt, points per trajectory, coalesced share), computed at the trajectory level; per-device comparison of turning angle.

**Go criterion.** |ρ(turning angle, median Δt)| < 0.5.

In [ ]:
covs = ['median_dt_ms', 'n_points', 'coalesced_share']
rows = []
for f in CORE_FEATURES:
    for c in covs:
        if c in feat and feat[c].notna().sum() > 3:
            rho, p = stats.spearmanr(feat[f], feat[c], nan_policy='omit'); rows.append({'feature': f, 'covariate': c, 'rho': rho, 'p': p})
sens = pd.DataFrame(rows)
display(sens.pivot(index='feature', columns='covariate', values='rho').round(2))
rho_turn_dt = sens.loc[(sens.feature == 'mean_abs_turn_deg') & (sens.covariate == 'median_dt_ms'), 'rho']
rho_turn_dt = float(rho_turn_dt.iloc[0]) if len(rho_turn_dt) else np.nan
print(f"ρ(turning angle, median Δt) = {rho_turn_dt:.2f}  (criterion |ρ| < {CRITERIA['max_abs_rho_turn_dt']})")
GO_SENSITIVITY = (abs(rho_turn_dt) < CRITERIA['max_abs_rho_turn_dt']) if np.isfinite(rho_turn_dt) else True

# Device comparison (self-report), descriptive only.
dev = audit[['session_key', 'mt_pointer_type_self_report']].rename(columns={'session_key': 'participant_id', 'mt_pointer_type_self_report': 'device'})
fd = feat.merge(dev, on='participant_id', how='left')
if fd['device'].notna().any():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, f in zip(axes, ['efficiency', 'mean_abs_turn_deg', 'median_dt_ms']):
        sns.boxplot(data=fd, x='device', y=f, ax=ax, color='#e2e8f0', fliersize=2)
        sns.stripplot(data=fd, x='device', y=f, ax=ax, alpha=.5, color='#1e3a8a', size=3)
        ax.set_title(FEATURE_LABELS[f]); ax.set_xlabel('self-reported device'); ax.set_ylabel('')
    plt.tight_layout(); plt.show()
    display(fd.groupby('device')[CORE_FEATURES + ['median_dt_ms']].median().round(3))

## Pilot decision summary

Each row is a pre-specified check with its criterion. This table is the thing to bring to the lab meeting; the escalation decision (Sigma-Lognormal features / a classifier) is only on the table if every check is a **GO**.

In [ ]:
summary = pd.DataFrame([
    {'check': 'Completion & devices', 'value': f'{complete_rate:.0%} complete of {len(audit)} expected', 'criterion': f"≥ {CRITERIA['min_complete_rate']:.0%}", 'go': GO_COMPLETE},
    {'check': 'Sampling quality', 'value': f'{pass_rate:.0%} of completed participants pass', 'criterion': f"≥ {CRITERIA['min_sampling_pass_rate']:.0%} (median ≥ {CRITERIA['min_points_median']} pts, Δt ≤ {CRITERIA['max_dt_median_ms']:.0f} ms)", 'go': GO_SAMPLING},
    {'check': 'Feature sanity', 'value': f'median efficiency {eff_med:.3f}', 'criterion': 'L increases with nominal length (ρ > .5); efficiency < .99', 'go': GO_FEATURES},
    {'check': 'Synthetic separation (type A)', 'value': f'separation {sepA:.2f}', 'criterion': f"≥ {CRITERIA['min_separation_typeA']:.2f}", 'go': GO_SEPARATION},
    {'check': 'Sampling sensitivity', 'value': f'ρ(turn, Δt) = {rho_turn_dt:.2f}', 'criterion': f"|ρ| < {CRITERIA['max_abs_rho_turn_dt']}", 'go': GO_SENSITIVITY},
])
summary['go'] = summary['go'].map({True: '✅ GO', False: '❌ NO-GO'})
display(summary.set_index('check'))
overall = summary['go'].str.contains('GO').all() and not summary['go'].str.contains('NO-GO').any()
print('OVERALL:', 'GO — the measure behaves sensibly; escalation can be discussed' if overall else 'NO-GO — keep the module exploratory or drop it')

# Export primary features and the full exclusion audit separately.
out_dir = os.path.join(CLEAN_DIR, 'derived'); os.makedirs(out_dir, exist_ok=True)
pp.to_csv(os.path.join(out_dir, 'mouse_qc_participant_summary.csv'), index=False)
feat.to_csv(os.path.join(out_dir, 'mouse_qc_trajectory_features.csv'), index=False)
sep.to_csv(os.path.join(out_dir, 'mouse_qc_synthetic_separation.csv'), index=False)
audit.to_csv(os.path.join(out_dir, 'mouse_qc_session_audit.csv'), index=False)
excluded_traj.to_csv(os.path.join(out_dir, 'mouse_qc_excluded_trajectories.csv'), index=False)
print('written →', out_dir)

## Caveats to keep in the write-up

- The completion denominator is all metadata sessions assigned/requesting QC; expected-but-missing summaries and asset failures are failures, not silently absent cases.
- Primary feature summaries require exactly one completed `trajectory_id` 1–8 and, when reported, `mt_n_trajectories_completed == 8`. Skipped, incomplete, duplicated-ID, and missing-ID data are reported separately.
- The pilot **cannot** estimate bot-detection accuracy: there are no labelled bots. Synthetic baselines show only that the *measurement* discriminates obviously non-human input.
- Coordinates are normalised to the task area, so speed is in side-units · s⁻¹; physical speed depends on `mt_area_side_px` and DPR.
- Turning angle is sampling-sensitive; report it alongside `median_dt_ms` and `mt_coalesced_supported`.
- Trackpads report `pointerType = mouse`; use the self-report for device.
- If `mt_geometry_version` changes, analyse versions separately.